In [ ]:
from nyc_taxi_trips_etl.utils.pipeline_params_parsing import parse_and_expand_year_months

year_months_to_extract = dbutils.widgets.get("year_months_to_extract")
catalog = dbutils.widgets.get("catalog")

In [ ]:
year_months = parse_and_expand_year_months(year_months_to_extract)

In [ ]:

trip_summary_df = spark.sql(f"""
    SELECT 
        s.year_month,
        tzl.borough,
        AVG(s.total_amount) AS avg_fare,
        COUNT(s.pickup_location_id) AS total_trips,
        ROUND(AVG(s.total_amount / s.trip_distance), 2) AS avg_cost_per_mile
    FROM {catalog}.silver.yellow_trip_data s
    LEFT JOIN {catalog}.bronze.taxi_zone_lookup_vw tzl -- TODO should it be moved to silver?
        ON s.pickup_location_id = tzl.location_id
    WHERE s.year_month IN ({','.join([f"'{ym}'" for ym in year_months])})
        AND s.trip_distance > 0
    GROUP BY s.year_month, tzl.borough
    ORDER BY s.year_month, tzl.borough
""")

In [ ]:
(trip_summary_df.write
    .mode("overwrite")
    .option("replaceWhere", f"year_month in ({','.join([f"'{ym}'" for ym in year_months])})")
    .saveAsTable(f"{catalog}.gold.trip_summary_per_borough")
)